In [9]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms, datasets
import torchvision.models as models

import sys
from tqdm import tqdm
import time

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [6]:
import requests
import os
import zipfile

DATA_URL = 'https://download.pytorch.org/tutorial/hymenoptera_data.zip'
DATA_PATH = os.path.join('.', '../content/data')
FILE_NAME = os.path.join(DATA_PATH, 'hymenoptera_data.zip')

if not os.path.isfile(FILE_NAME):
    print("Downloading the data...")
    os.makedirs('../content/data', exist_ok=True)
    with requests.get(DATA_URL) as req:
        with open(FILE_NAME, 'wb') as f:
            f.write(req.content)
        if 200 <= req.status_code < 300:
            print("Download complete!")
        else:
            print("Download failed!")
else:
    print(FILE_NAME, "already exists, skipping download...")

with zipfile.ZipFile(FILE_NAME, 'r') as zip_ref:
    print("Unzipping...")
    zip_ref.extractall('data')

DATA_PATH = os.path.join(DATA_PATH, 'hymenoptera_data')

Download complete!
Unzipping...


In [5]:
!wget https://download.pytorch.org/tutorial/hymenoptera_data.zip
!unzip hymenoptera_data.zip

"wget" не является внутренней или внешней
командой, исполняемой программой или пакетным файлом.
"unzip" не является внутренней или внешней
командой, исполняемой программой или пакетным файлом.


In [7]:
transforms_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transforms_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [10]:
train_data = datasets.ImageFolder('./drive/MyDrive/Colab_Notebooks/content/hymenoptera_data/train') if 'google.colab' in sys.modules \
    else datasets.ImageFolder('/Users/SashaCurry/PyCharmMiscProject/content/hymenoptera_data/train', transform=transforms_train)
val_data = datasets.ImageFolder('./drive/MyDrive/Colab_Notebooks/content/hymenoptera_data/val') if 'google.colab' in sys.modules \
    else datasets.ImageFolder('/Users/SashaCurry/PyCharmMiscProject/content/hymenoptera_data/val', transform=transforms_train)

In [11]:
class_names = train_data.classes
class_names

['ants', 'bees']

In [12]:
print(len(train_data))
print(len(val_data))

244
153


In [13]:
train_loader =  DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)

# Обучение с нуля, без Transfer Learning

## Вариант 1

1. Модель не обучена
2. Меняем только последний слой

In [14]:
model = models.vgg11()
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [15]:
model.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [18]:
model.classifier[6] = nn.Linear(4096, out_features=2)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [20]:
model = model.to(device)

loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

## Вариант 2

1. Модель не обучена
2. Меняем весь классификатор

In [22]:
model = models.vgg11()
model.classifier = nn.Linear(512*7*7, 2)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [23]:
model = model.to(device)

loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

## Тренировка модели

In [24]:
EPOCHS = 10
train_loss = []
train_acc = []
val_loss = []
val_acc = []
lr_list = []
best_loss = None
best_epoch = None
count = 0

start = time.time()
# Цикл обучения
for epoch in range(EPOCHS):
    # Тренировка модели
    running_train_loss = []
    true_answer = 0
    mean_train_loss = 0
    # train_loop = tqdm(train_loader, leave=False) # создание progress bar

    model.train()
    # for x, targets in train_loop:
    for x, targets in train_loader:
        # Данные
        # (batch_size, 1, 28, 28) -> (batch_size, 784)
        # x = x.reshape(-1, 28*28).to*device)
        x = x.to(device)
        # (batch_size, int) -> (batch_size, 10), dtype=float32
        targets = targets.reshape(-1).to(torch.int32)
        targets = torch.eye(2)[targets].to(device)

        # Прямой проход + расчет ошибки модели
        pred = model(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss) / len(running_train_loss)

        true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # train_loop.set_description(f'Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}')

    # Расчет значения метрики
    running_train_acc = true_answer / len(train_data)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    model.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 28, 28) -> (batch_size, 784)
            # x = x.reshape(-1, 28*28).to*device)
            x = x.to(device)
            # (batch_size, int) -> (batch_size, 10), dtype=float32
            targets = targets.reshape(-1).to(torch.int32)
            targets = torch.eye(2)[targets].to(device)

            # Прямой проход + расчет ошибки модели
            pred = model(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss) / len(running_val_loss)

            true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # Расчет значения метрики
        running_val_acc = true_answer / len(val_data)

        # Сохранение значения функции потерь и метрики
        val_loss.append(mean_val_loss)
        val_acc.append(running_val_acc)

        lr_scheduler.step(mean_val_loss)
        lr = opt.param_groups[0]['lr']
        lr_list.append(lr)

        print(f'Epoch [{epoch + 1}/{EPOCHS}]: train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr:.4f}')

        # if best_loss is None:
        #     best_loss = mean_val_loss
        #     best_epoch = epoch
        #
        # if mean_val_loss < best_loss:
        #     # if os.path.exists(f'model_state_dict_epoch_{best_epoch}.pt'):
        #     #     os.remove(f'model_state_dict_epoch_{best_epoch}.pt')
        #
        #     best_loss = mean_val_loss
        #     best_epoch = epoch + 1
        #
        #     checkpoint = {
        #         'state_model': model.state_dict(),
        #         'state_opt': opt.state_dict(),
        #         'state_lr_scheduler': lr_scheduler.state_dict(),
        #         'loss': {
        #             'train_loss': train_loss,
        #             'val_loss': val_loss,
        #             'best_loss': best_loss
        #         },
        #         'metric': {
        #             'train_acc': train_acc,
        #             'val_acc': val_acc,
        #         },
        #         'lr': lr_list,
        #         'epoch': {
        #             'EPOCHS': EPOCHS,
        #             'save_epoch': epoch
        #         }
        #     }
        #
        #     torch.save(checkpoint, f'model_state_dict_epoch_{best_epoch}.pt')
        #     print(
        #         f'На эпохе {epoch + 1} сохранена модель со значением функции потерь на валидации = {mean_val_loss:.4f}')
        #
        # if earlystopping(mean_val_loss):
        #     print(f'\033[31mОбучение остановлено на {epoch + 1} эпохе.\033[m')
        #     break

time_elapsed = time.time() - start
print(f'Время обучения {EPOCHS} эпох: {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')

Epoch [1/10]: train_loss=2.9625, train_acc=0.4754, val_loss=0.6948, val_acc=0.4575, lr=0.0010
Epoch [2/10]: train_loss=0.6924, train_acc=0.5123, val_loss=0.6843, val_acc=0.5425, lr=0.0010
Epoch [3/10]: train_loss=0.6917, train_acc=0.5246, val_loss=0.6963, val_acc=0.4575, lr=0.0010
Epoch [4/10]: train_loss=0.6962, train_acc=0.4631, val_loss=0.6932, val_acc=0.4902, lr=0.0010
Epoch [5/10]: train_loss=0.6925, train_acc=0.5041, val_loss=0.6862, val_acc=0.5490, lr=0.0010
Epoch [6/10]: train_loss=0.6932, train_acc=0.5287, val_loss=0.6950, val_acc=0.4575, lr=0.0010
Epoch [7/10]: train_loss=0.6927, train_acc=0.5082, val_loss=0.6942, val_acc=0.4837, lr=0.0010
Epoch [8/10]: train_loss=0.7047, train_acc=0.6107, val_loss=0.6668, val_acc=0.5490, lr=0.0010
Epoch [9/10]: train_loss=0.7126, train_acc=0.5082, val_loss=0.6983, val_acc=0.4575, lr=0.0010
Epoch [10/10]: train_loss=0.6959, train_acc=0.5041, val_loss=0.6937, val_acc=0.4575, lr=0.0010
Время обучения 10 эпох: 0m 28s


# Обучение с методом Transfer Learning

## Вариант 1

1. Модель обучена
2. Не замораживаем обученные параметры
3. Меняем последний слой

In [25]:
model = models.vgg11(weights='DEFAULT')
model.classifier[6] = nn.Linear(4096, 2)
model

Downloading: "https://download.pytorch.org/models/vgg11-8a719046.pth" to C:\Users\SashaCurry/.cache\torch\hub\checkpoints\vgg11-8a719046.pth


100%|██████████| 507M/507M [00:47<00:00, 11.2MB/s] 


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [26]:
model = model.to(device)

loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

## Вариант 2

1. Модель обучена
2. Не замораживаем обученные параметры
3. Меняем весь классификатор

In [31]:
model = models.vgg11(weights='DEFAULT')
model.classifier = nn.Linear(512*7*7, 2)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [32]:
model = model.to(device)

loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

## Вариант 3

1. Модель обучена
2. Замораживаем обученные параметры
3. Меняем весь классификатор

In [34]:
model = models.vgg11(weights='DEFAULT')

for parameter in model.parameters():
    parameter.requires_grad = False

model.classifier = nn.Linear(512*7*7, 2)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [37]:
for name, layer in model.named_modules():
    if isinstance(layer, (nn.Conv2d, nn.Linear)):
        print(name)
        for i, param in enumerate(layer.parameters()):
            if i == 0:
                print(f"    weights.requiras_grad = {param.requires_grad}")
            else:
                print(f"    bias.requiras_grad    = {param.requires_grad}", end="\n\n")

features.0
    weights.requiras_grad = False
    bias.requiras_grad    = False

features.3
    weights.requiras_grad = False
    bias.requiras_grad    = False

features.6
    weights.requiras_grad = False
    bias.requiras_grad    = False

features.8
    weights.requiras_grad = False
    bias.requiras_grad    = False

features.11
    weights.requiras_grad = False
    bias.requiras_grad    = False

features.13
    weights.requiras_grad = False
    bias.requiras_grad    = False

features.16
    weights.requiras_grad = False
    bias.requiras_grad    = False

features.18
    weights.requiras_grad = False
    bias.requiras_grad    = False

classifier
    weights.requiras_grad = True
    bias.requiras_grad    = True



In [38]:
model = model.to(device)

loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.classifier.parameters(), lr=0.001)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

## Вариант 4

Finetuning

In [42]:
model.features[13:]

Sequential(
  (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (14): ReLU(inplace=True)
  (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (16): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (17): ReLU(inplace=True)
  (18): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (19): ReLU(inplace=True)
  (20): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
)

In [46]:
for parameter in model.features[13:].parameters():
    parameter.requires_grad = True

In [47]:
for name, layer in model.named_modules():
    if isinstance(layer, (nn.Conv2d, nn.Linear)):
        print(name)
        for i, param in enumerate(layer.parameters()):
            if i == 0:
                print(f"    weights.requiras_grad = {param.requires_grad}")
            else:
                print(f"    bais.requiras_grad    = {param.requires_grad}", end="\n\n")

features.0
    weights.requiras_grad = False
    bais.requiras_grad    = False

features.3
    weights.requiras_grad = False
    bais.requiras_grad    = False

features.6
    weights.requiras_grad = False
    bais.requiras_grad    = False

features.8
    weights.requiras_grad = False
    bais.requiras_grad    = False

features.11
    weights.requiras_grad = False
    bais.requiras_grad    = False

features.13
    weights.requiras_grad = True
    bais.requiras_grad    = True

features.16
    weights.requiras_grad = True
    bais.requiras_grad    = True

features.18
    weights.requiras_grad = True
    bais.requiras_grad    = True

classifier
    weights.requiras_grad = True
    bais.requiras_grad    = True



In [48]:
model = model.to(device)

loss_model = nn.CrossEntropyLoss()
opt = torch.optim.Adam(
    [
        {'params': model.features[13:].parameters(), 'lr': 0.000001},
        {'params': model.classifier.parameters()}
    ],
    lr=0.0001
)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5)

In [49]:
EPOCHS = 10
train_loss = []
train_acc = []
val_loss = []
val_acc = []
lr_list = []
best_loss = None
best_epoch = None
count = 0

start = time.time()
# Цикл обучения
for epoch in range(EPOCHS):
    # Тренировка модели
    running_train_loss = []
    true_answer = 0
    mean_train_loss = 0
    # train_loop = tqdm(train_loader, leave=False) # создание progress bar

    model.train()
    # for x, targets in train_loop:
    for x, targets in train_loader:
        # Данные
        # (batch_size, 1, 28, 28) -> (batch_size, 784)
        # x = x.reshape(-1, 28*28).to*device)
        x = x.to(device)
        # (batch_size, int) -> (batch_size, 10), dtype=float32
        targets = targets.reshape(-1).to(torch.int32)
        targets = torch.eye(2)[targets].to(device)

        # Прямой проход + расчет ошибки модели
        pred = model(x)
        loss = loss_model(pred, targets)

        # Обратный проход
        opt.zero_grad()
        loss.backward()
        # Шаг оптимизации
        opt.step()

        running_train_loss.append(loss.item())
        mean_train_loss = sum(running_train_loss) / len(running_train_loss)

        true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # train_loop.set_description(f'Epoch [{epoch+1}/{EPOCHS}], train_loss={mean_train_loss:.4f}')

    # Расчет значения метрики
    running_train_acc = true_answer / len(train_data)

    # Сохранение значения функции потерь и метрики
    train_loss.append(mean_train_loss)
    train_acc.append(running_train_acc)

    # Проверка модели (валидация)
    model.eval()
    with torch.no_grad():
        running_val_loss = []
        true_answer = 0
        for x, targets in val_loader:
            # Данные
            # (batch_size, 1, 28, 28) -> (batch_size, 784)
            # x = x.reshape(-1, 28*28).to*device)
            x = x.to(device)
            # (batch_size, int) -> (batch_size, 10), dtype=float32
            targets = targets.reshape(-1).to(torch.int32)
            targets = torch.eye(2)[targets].to(device)

            # Прямой проход + расчет ошибки модели
            pred = model(x)
            loss = loss_model(pred, targets)

            running_val_loss.append(loss.item())
            mean_val_loss = sum(running_val_loss) / len(running_val_loss)

            true_answer += (pred.argmax(dim=1) == targets.argmax(dim=1)).sum().item()

        # Расчет значения метрики
        running_val_acc = true_answer / len(val_data)

        # Сохранение значения функции потерь и метрики
        val_loss.append(mean_val_loss)
        val_acc.append(running_val_acc)

        lr_scheduler.step(mean_val_loss)
        lr = opt.param_groups[0]['lr']
        lr_list.append(lr)

        print(
            f'Epoch [{epoch + 1}/{EPOCHS}]: train_loss={mean_train_loss:.4f}, train_acc={running_train_acc:.4f}, val_loss={mean_val_loss:.4f}, val_acc={running_val_acc:.4f}, lr={lr:.4f}')

        # if best_loss is None:
        #     best_loss = mean_val_loss
        #     best_epoch = epoch
        #
        # if mean_val_loss < best_loss:
        #     # if os.path.exists(f'model_state_dict_epoch_{best_epoch}.pt'):
        #     #     os.remove(f'model_state_dict_epoch_{best_epoch}.pt')
        #
        #     best_loss = mean_val_loss
        #     best_epoch = epoch + 1
        #
        #     checkpoint = {
        #         'state_model': model.state_dict(),
        #         'state_opt': opt.state_dict(),
        #         'state_lr_scheduler': lr_scheduler.state_dict(),
        #         'loss': {
        #             'train_loss': train_loss,
        #             'val_loss': val_loss,
        #             'best_loss': best_loss
        #         },
        #         'metric': {
        #             'train_acc': train_acc,
        #             'val_acc': val_acc,
        #         },
        #         'lr': lr_list,
        #         'epoch': {
        #             'EPOCHS': EPOCHS,
        #             'save_epoch': epoch
        #         }
        #     }
        #
        #     torch.save(checkpoint, f'model_state_dict_epoch_{best_epoch}.pt')
        #     print(
        #         f'На эпохе {epoch + 1} сохранена модель со значением функции потерь на валидации = {mean_val_loss:.4f}')
        #
        # if earlystopping(mean_val_loss):
        #     print(f'\033[31mОбучение остановлено на {epoch + 1} эпохе.\033[m')
        #     break

time_elapsed = time.time() - start
print(f'Время обучения {EPOCHS} эпох: {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')

Epoch [1/10]: train_loss=0.0035, train_acc=1.0000, val_loss=0.8138, val_acc=0.8954, lr=0.0000
Epoch [2/10]: train_loss=0.0404, train_acc=0.9877, val_loss=0.5841, val_acc=0.9085, lr=0.0000
Epoch [3/10]: train_loss=0.0057, train_acc=0.9959, val_loss=0.5864, val_acc=0.9412, lr=0.0000
Epoch [4/10]: train_loss=0.0070, train_acc=0.9959, val_loss=0.7386, val_acc=0.9085, lr=0.0000
Epoch [5/10]: train_loss=0.0306, train_acc=0.9959, val_loss=0.6460, val_acc=0.9281, lr=0.0000
Epoch [6/10]: train_loss=0.0029, train_acc=1.0000, val_loss=0.7419, val_acc=0.9020, lr=0.0000
Epoch [7/10]: train_loss=0.0036, train_acc=0.9959, val_loss=0.6363, val_acc=0.9216, lr=0.0000
Epoch [8/10]: train_loss=0.0024, train_acc=1.0000, val_loss=0.7973, val_acc=0.9085, lr=0.0000
Epoch [9/10]: train_loss=0.0029, train_acc=1.0000, val_loss=0.6550, val_acc=0.9281, lr=0.0000
Epoch [10/10]: train_loss=0.0073, train_acc=0.9959, val_loss=0.6405, val_acc=0.9216, lr=0.0000
Время обучения 10 эпох: 0m 24s
